# Vave - Bronze Table Ingestion (dbt-Triggered)

This notebook ingests files from internal Volume storage into the Bronze Delta table.

**Orchestration Model:**
- dbt handles file population from external → internal storage
- dbt triggers this notebook via event/workflow
- This notebook processes files and inserts into bronze table

**Prerequisites:** 
1. dbt has copied files to internal storage
2. Required schemas and tables exist (created by dbt or Setup notebook)

In [0]:
%run /Workspace/Shared/Vave_Config_Params

In [0]:
from pyspark.sql import functions as F

# Parse file paths (comma-separated from dbt)
files_to_process = [f.strip() for f in fil_pth_str.split(",") if f.strip()]

print("="*60)
print("=== Bronze Ingestion - dbt Triggered ===")
print("="*60)
print(f"\n📊 Target Table: {brz_tbl}")
print(f"📝 File Type: {fil_typ}")
print(f"📁 Files to Process: {len(files_to_process)}")
if bch_id:
    print(f"🔖 Batch ID: {bch_id}")

if not files_to_process:
    raise ValueError("No file paths provided - dbt must pass file_paths parameter")

=== Bronze Ingestion - dbt Triggered ===

📊 Target Table: main.bronze.vave_eventsfeed
📝 File Type: csv
📁 Files to Process: 0



# Vave - Geo-Spatial Risk Aggregation <br> Config - Parameters
This is a central Params notebook <br>
This allow the call via other notebooks to reuse values set here<br> 

In a real world scenario, these params would mostly be passed by the orchestrator

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-7202909518971618>, line 16
     13     print(f"🔖 Batch ID: {bch_id}")
     15 if not files_to_process:
---> 16     raise ValueError("No file paths provided - dbt must pass file_paths parameter")

ValueError: No file paths provided - dbt must pass file_paths parameter

In [0]:
print("\n=== Ensuring Bronze Table Exists ===")

# Define bronze table schema
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {brz_tbl}
(
  event_id         STRING,
  event_timestamp  TIMESTAMP,
  latitude         DOUBLE,
  longitude        DOUBLE,
  risk_score       DOUBLE,
  source_file_name STRING,
  ingestion_ts     TIMESTAMP,
  dbt_run_id       STRING
)
USING DELTA
COMMENT 'Bronze layer - Raw event data with minimal transformations'
""")

print(f"✓ Bronze table ready: {brz_tbl}")

In [0]:
# CELL 4 - Ingest Files (dbt tells us exactly what to process)
print(f"\n=== Ingesting {len(files_to_process)} files ===")

# Build file paths string for read_files
file_paths_sql = "', '".join(files_to_process)

# Use batch_id for traceability
batch_id_value = batch_id if batch_id else "manual"

# Create and insert in one SQL statement
print("Reading and inserting files...")
spark.sql(f"""
INSERT INTO {brz_tbl}
SELECT
  event_id,
  CAST(event_timestamp AS TIMESTAMP) AS event_timestamp,
  CAST(latitude   AS DOUBLE)         AS latitude,
  CAST(longitude  AS DOUBLE)         AS longitude,
  CAST(risk_score AS DOUBLE)         AS risk_score,
  _metadata.file_path                AS source_file_name,
  current_timestamp()                AS ingestion_ts,
  '{batch_id_value}'                 AS batch_id
FROM read_files('{file_paths_sql}',  format => '{fil_typ}',  header => true)
""")

# Get count of what we just inserted
inserted_count = spark.sql(f""" SELECT COUNT(*) as cnt 
                                FROM {brz_tbl}
                                WHERE batch_id = '{batch_id_value}'
""").collect()[0]['cnt']

print(f"\n✓ Inserted {inserted_count:,} records")
print(f"✓ Processed {len(files_to_process)} files")

=== Injestion File Config ===
fil_typ:    csv
fil_nm:     vave_eventsfeed

=== Catalog ===
cat: main

=== Schema ===
scc: control
scr: _raw
scb: bronze
scs: silver
scg: gold
con: external_data

=== Volume Folders ===
raw_fld: vave_raw
ext_fld: pseudoexternalrepo

=== Computed Paths ===
raw_path_ext: /Volumes/main/_raw/pseudoexternalrepo
raw_path_int: /Volumes/main/_raw/external_data/vave_raw

=== Log tables ===
log_tbl:            main.control.ingested_files
brz_tbl_control:    main.control.processed_files

=== Medallion tables ===
brz_tbl: main.bronze.vave_eventsfeed
sil_tbl: main.silver.vave_eventsfeed
gld_tbl: main.gold.vave_eventsfeed
✓ Parameters loaded


In [0]:
print("\n" + "="*60)
print("=== Bronze Table Validation ===")
print("="*60)

try:
    bronze_df = spark.table(bronze_table)
    
    # Total records
    total_records = bronze_df.count()
    print(f"\n📊 Total Records in Bronze: {total_records:,}")
    
    # This run's records (if run_id provided)
    if run_id:
        this_run_count = bronze_df.filter(F.col("dbt_run_id") == run_id).count()
        print(f"📊 Records from this run ({run_id}): {this_run_count:,}")
    
    # Distinct source files
    distinct_sources = bronze_df.select("source_file_name").distinct().count()
    print(f"📄 Distinct Source Files: {distinct_sources}")
    
    # Latest ingestion
    latest = bronze_df.agg(F.max("ingestion_ts")).collect()[0][0]
    print(f"🕐 Latest Ingestion: {latest}")
    
    # Sample data
    print(f"\n=== Sample Records (Latest Ingestion) ===")
    display(
        bronze_df
        .orderBy(F.col("ingestion_ts").desc())
        .limit(5)
    )
    
except Exception as e:
    print(f"⚠️  Validation error: {e}")

print("\n" + "="*60)
print("✓ Bronze ingestion complete")
print("="*60)